Pour pouvoir executer un code de ce genre il faut faire setter une variable d'environnement dans powershell si on utilise windows
$env:OLLAMA_NUM_PARALLEL = 4


In [ ]:
from concurrent.futures import ThreadPoolExecutor
import time

client = OpenAI(
    base_url="http://127.0.0.1:11434/v1",
    api_key="ollama"
)

def call(prompt: str) -> str:
    resp = client.chat.completions.create(
        model="phi4-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=1,
        temperature=0
    )
    return resp.choices[0].message.content.strip()

# --- Test ---
prompts = ["say hi", "say hello", "say good morning", "say thanks"]

# 1. Séquentiel (référence)
t0 = time.time()
for p in prompts:
    call(p)
t_seq = time.time() - t0

# 2. Concurrent (4 threads)
t0 = time.time()
with ThreadPoolExecutor(max_workers=4) as pool:
    list(pool.map(call, prompts))
t_par = time.time() - t0

print(f"Séquentiel : {t_seq:.2f}s")
print(f"Concurrent : {t_par:.2f}s")
print(f"Gain       : {t_seq / t_par:.1f}x")   